# Counting CNN (Convolutional Neural Network) parameters

_Deep Learning_

**How many weights a convolutional layer holds, and what a neuron can 'see'.**

A convolutional layer learns the numbers inside its filters. Counting them tells you the layer's size.

     Each filter is $F\times F$ wide and goes through all $C$ input channels (like red, green, blue). Plus one bias each.

---

This notebook is a practice scaffold for the **Counting CNN (Convolutional Neural Network) parameters** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

cnn = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3*(3*3*3)+16... per filter
    nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
)

# params for conv layer 1: out * (in * fh * fw) + out (biases)
manual = 16 * (3 * 3 * 3) + 16
print("layer-1 params (by hand):", manual)        # 448

for name, layer in [("conv1", cnn[0]), ("conv2", cnn[2])]:
    n = sum(p.numel() for p in layer.parameters())
    print(name, "params:", n)
print("total:", sum(p.numel() for p in cnn.parameters()))

## Visualize the data & results

_For real 8x8 digit images, why is a convolutional layer so much cheaper than a fully-connected one?_

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
h, w = digits.images[0].shape            # real image = 8 x 8, 1 channel

conv1 = 16 * (1 * 3 * 3) + 16            # 160
conv2 = 32 * (16 * 3 * 3) + 32           # 4640
fc = (h * w * 1) * 128 + 128             # 8320 over the flattened image

labels = ["conv1 (1->16, 3x3)", "conv2 (16->32, 3x3)", "FC (64->128)"]
values = [conv1, conv2, fc]
colors = ["#7ee787", "#4ea1ff", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("Parameter count for an 8x8 digit input: conv vs FC")
plt.yscale("log")
plt.xticks(rotation=15)
plt.show()